In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 74.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a4e45b7d-cd4f-44d1-93d7-4b690114b1a6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e29a98536a5102024ca68e1b3b15c700fbadc4254cdc6805c8673067d57d85df




Accessing as Nestor-Dzadzamia

Initialized MLflow to track repo "Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection"

Repository Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection initialized!

In [5]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

print(train_transaction.shape)
print(train_identity.shape)

(590540, 394)
(144233, 41)


# Merge data

In [6]:
df = train_transaction.merge(train_identity, on='TransactionID', how='left')
print(f"Merged shape: {df.shape}")

y = df['isFraud']
X = df.drop(columns=['isFraud', 'TransactionID'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Fraud ratio: {y_train.mean():.4f}")

Merged shape: (590540, 434)
X_train shape: (472432, 432)
X_test shape: (118108, 432)
Fraud ratio: 0.0350


# Creating Preprocessor class

In [7]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_thresh=0.5, corr_thresh=0.9):
        self.null_thresh = null_thresh
        self.corr_thresh = corr_thresh

    def _feature_engineering(self, X):
        X = X.copy()
        X['log_TransactionAmt'] = np.log1p(X['TransactionAmt'])
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7
        return X

    def fit(self, X, y=None):
        X = self._feature_engineering(X)

        # drop high null columns
        self.cols_to_drop_ = X.columns[X.isnull().mean() > self.null_thresh].tolist()
        X = X.drop(columns=self.cols_to_drop_)

        # drop correlated columns
        corr_matrix = X.select_dtypes(include=np.number).corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.corr_cols_to_drop_ = [col for col in upper.columns if any(upper[col] > self.corr_thresh)]
        X = X.drop(columns=self.corr_cols_to_drop_)

        # separate categorial and numerical columns
        self.cat_cols_ = [col for col in X.columns if X[col].dtype == 'object']
        self.num_cols_ = [col for col in X.columns if X[col].dtype != 'object']
        
        self.num_imputer_ = SimpleImputer(strategy='median')
        self.num_imputer_.fit(X[self.num_cols_])
        
        self.cat_imputer_ = SimpleImputer(strategy='most_frequent')
        self.cat_imputer_.fit(X[self.cat_cols_])

        # encode and scale
        self.encoder_ = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        X_cat_imputed = self.cat_imputer_.transform(X[self.cat_cols_])
        self.encoder_.fit(X_cat_imputed)
        self.scaler_ = StandardScaler()
        
        X_num_imputed = self.num_imputer_.transform(X[self.num_cols_])
        self.scaler_.fit(X_num_imputed)
        
        return self

    def transform(self, X):
        X = self._feature_engineering(X)
        
        X = X.drop(columns=self.cols_to_drop_)
        X = X.drop(columns=self.corr_cols_to_drop_)
        
        X_num = self.num_imputer_.transform(X[self.num_cols_])
        X_cat = self.cat_imputer_.transform(X[self.cat_cols_])
        
        X_cat_encoded = self.encoder_.transform(X_cat)
        X_num_scaled = self.scaler_.transform(X_num)
       
        X_final = np.hstack([X_num_scaled, X_cat_encoded])
        
        return X_final

# Pipeline/experiments with mlflow logging

In [8]:
from sklearn.ensemble import RandomForestClassifier

mlflow.set_experiment("RandomForest_Training")

experiments = [
    {"null_thresh": 0.5, "corr_thresh": 0.9, "n_estimators": 100, "max_depth": None},
    {"null_thresh": 0.5, "corr_thresh": 0.9, "n_estimators": 100, "max_depth": 10},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "n_estimators": 200, "max_depth": 10},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "n_estimators": 200, "max_depth": 15},
]

for exp in experiments:
    pipe = Pipeline(steps=[
        ('preprocessor', FraudPreprocessor(null_thresh=exp["null_thresh"], corr_thresh=exp["corr_thresh"])),
        ('model', RandomForestClassifier(
            n_estimators=exp["n_estimators"],
            max_depth=exp["max_depth"],
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    pipe.fit(X_train, y_train)
    
    with mlflow.start_run(run_name=f"RF_n={exp['n_estimators']}_depth={exp['max_depth']}_null={exp['null_thresh']}"):
        mlflow.log_params({**exp, "class_weight": "balanced"})
        
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
        
        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_proba)
        mlflow.log_metric("train_roc_auc", train_roc_auc)
        mlflow.log_metric("test_roc_auc", test_roc_auc)
        
        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)
        
        cm = confusion_matrix(y_test, y_pred)
        mlflow.log_metric("tn", cm[0,0])
        mlflow.log_metric("fp", cm[0,1])
        mlflow.log_metric("fn", cm[1,0])
        mlflow.log_metric("tp", cm[1,1])
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc')
        mlflow.log_metric("cv_mean_roc_auc", cv_scores.mean())
        mlflow.log_metric("cv_std_roc_auc", cv_scores.std())
        
        mlflow.sklearn.log_model(pipe, artifact_path="random_forest_pipeline")
        
        print(f"RF n={exp['n_estimators']} depth={exp['max_depth']} null={exp['null_thresh']} → Train: {train_roc_auc:.4f} Test: {test_roc_auc:.4f} CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

2026/05/06 12:12:47 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_Training' does not exist. Creating a new experiment.
2026/05/06 12:28:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 12:28:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF n=100 depth=None null=0.5 → Train: 1.0000 Test: 0.9411 CV: 0.9368 ± 0.0031
🏃 View run RF_n=100_depth=None_null=0.5 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/aa82951e1d9b4568bbd7012cfab2a00a
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


2026/05/06 12:41:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 12:41:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF n=100 depth=10 null=0.5 → Train: 0.9015 Test: 0.8855 CV: 0.8877 ± 0.0028
🏃 View run RF_n=100_depth=10_null=0.5 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/c14379a95a2b455bb37e3cedfae10a8c
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


2026/05/06 12:59:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 12:59:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF n=200 depth=10 null=0.7 → Train: 0.9058 Test: 0.8892 CV: 0.8897 ± 0.0034
🏃 View run RF_n=200_depth=10_null=0.7 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/070842f8c68b459c952900a579a8cbdf
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


2026/05/06 13:19:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 13:19:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF n=200 depth=15 null=0.7 → Train: 0.9616 Test: 0.9193 CV: 0.9156 ± 0.0033
🏃 View run RF_n=200_depth=15_null=0.7 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/38aa985125a145fd9e86534a317db033
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
